<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Import Modules
    <\h1>
</span>

In [1]:
%config Completer.use_jedi = False

import platform
import pandas as pd
import dash
import dash_core_components as dcc
import dash_html_components as html
from jupyter_dash import JupyterDash
import dash_table
from dash.dependencies import Input, Output, State

<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Read Raw Data
    <\h1>
</span>

In [2]:
if platform.system() == "Linux":
    PROJECT_PATH = "/mnt/c/Users/POOYA/Dropbox/Work/HydroTech/Landscape_Evapotranspiration/"
else:
    PROJECT_PATH = "C:/Users/POOYA/Dropbox/Work/HydroTech/Landscape_Evapotranspiration/"

raw_data = pd.read_excel(PROJECT_PATH + "assets/input/Raw_Data_20210216.xlsx", 
                         engine = "openpyxl")

<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Create Dashboard
    <\h1>
</span>

In [ ]:
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

app = JupyterDash(__name__, external_stylesheets=external_stylesheets)


app.layout = html.Div([
    
    html.Div([

        html.Div([
            html.H5("Select A Lakeh:"),
            dcc.Dropdown(
                id="dropdown_lakeh",
                options=[{'label': i, 'value': i} for i in raw_data["نام لکه"].unique()],
                placeholder= "Select A Lakeh ..."
            ),
        ], style={'width': '49%',
                  'display': 'inline-block'}
        ),
        
        html.Div([
            html.H5("Select A Peyman:"),
            dcc.Dropdown(
                id="dropdown_peyman",
                placeholder= "Select A Peyman ..."
            ),
        ], style={'width': '49%',
                  'display': 'inline-block'}
        )
    ], style={'border': 'thin lightgrey solid',
              'backgroundColor': 'rgb(250, 250, 250)',
              'padding': '10px 5px'}
    ),

    html.Div([
        dash_table.DataTable(id='table-container',
                             columns=[{"name": i, "id": i} for i in raw_data.columns],
                             filter_action="native",
                             sort_action="native",
                             sort_mode='multi',
                             selected_rows=[],
                             page_action='native',
                             page_current= 0,
                             page_size= 50)
    ])
])


@app.callback(Output('dropdown_peyman', 'options'),
              Input('dropdown_lakeh', 'value'))
def update_dropdown_peyman(dropdown_lakeh):
    return [{'label': i, 'value': i} for i in raw_data[raw_data["نام لکه"] == dropdown_lakeh]["پیمان"].unique()]

@app.callback(Output('table-container', 'data'),
              Input('dropdown_lakeh', 'value'),
              Input('dropdown_peyman', 'value'))
def display_table(dropdown_lakeh, dropdown_peyman):
    if dropdown_lakeh is None and dropdown_peyman is None:
        return raw_data.to_dict('records')
    df = raw_data[raw_data["نام لکه"] == dropdown_lakeh]
    df = df[df["پیمان"] == dropdown_peyman]    
    return df.to_dict('records')

app.run_server(mode="inline")

In [10]:
import pandas as pd
import dash
import os
from dash.dependencies import Input, Output
import dash_core_components as dcc
import dash_html_components as html
import dash_table as dt


app = JupyterDash(__name__)
app.css.config.serve_locally = True

def generate_table(raw_data):
    
    df = raw_data

    return dt.DataTable(
        id="table",
        sort_action="native",
        filter_action="native",
        row_deletable=False,
        style_data={"whiteSpace": "normal"},
        style_cell={
            "padding": "5px",
            "midWidth": "0px",
            "width": "5%",
            "textAlign": "center",
            "border": "white",
            'whiteSpace': 'normal',
            'height': 'auto'
        },
        page_size=30,
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict("rows"),
    )

def generate_frontpage():
    return html.Div(
        id="las-header",
        children=[
            html.Img(id="las-logo", src=app.get_asset_url("mashhad.png")),
#             html.Div(
#                 id="las-header-text",
#                 children=[
#                     html.H1("Dataset"),
#                 ],
#             ),
            html.Img(id="dash-logo", src=app.get_asset_url("hydrotech.png")),
        ],
    )


app.layout = html.Div(
    [
        html.Div(id="frontpage", className="page", children=generate_frontpage()),
        html.Div(
            className="section",
            children=[
                html.Div(className="section-title", children=" بانک داده های خام"),
                
                html.Div([
                    html.Div([
                        html.H3("انتخاب لکه"),
                        dcc.Dropdown(
                            id="dropdown_lakeh",
                            options=[{'label': i, 'value': i} for i in raw_data["نام لکه"].unique()],
                            placeholder= "Select A Lakeh ..."
                        ),
                    ], style={'width': '49%',
                              'display': 'inline-block'}
                    ),

                    html.Div([
                        html.H3("انتخاب پیمان"),
                        dcc.Dropdown(
                            id="dropdown_peyman",
                            placeholder= "Select A Peyman ..."
                        ),
                    ], style={'width': '49%',
                              'display': 'inline-block'}
                    )
                ], style={'border': 'thin lightgrey solid',
                          'backgroundColor': 'rgb(250, 250, 250)',
                          'padding': '10px 5px'}
                ),  
                              
                
                html.Div(
                    className="page",
                    children=[
                        html.Div(id="las-table"),
                    ],
                ),
            ],
        )
    ]
)



@app.callback(Output('dropdown_peyman', 'options'),
              Input('dropdown_lakeh', 'value'))
def update_dropdown_peyman(dropdown_lakeh):
    return [{'label': i, 'value': i} for i in raw_data[raw_data["نام لکه"] == dropdown_lakeh]["پیمان"].unique()]


@app.callback(Output('las-table', 'children'),
              Input('dropdown_lakeh', 'value'),
              Input('dropdown_peyman', 'value'))
def display_table(dropdown_lakeh, dropdown_peyman):
    if dropdown_lakeh is None and dropdown_peyman is None:
        return generate_table(raw_data)
    df = raw_data[raw_data["نام لکه"] == dropdown_lakeh]
    df = df[df["پیمان"] == dropdown_peyman]    
    return generate_table(df)


app.run_server(mode="external")

Dash app running on http://127.0.0.1:8050/
